# 04 · 파형 특징 추출

박동 하나에서 **23개**를 재고, 환자 단위로 요약해 **43개**를 만든다.

## 박동 단위 23개

| 계열 | 개수 | 특징 |
|---|---|---|
| 타이밍 | 11 | IBI · CT · LVET · CT/LVET · CT/IBI · LVET/IBI · dT · W25 · W50 · W75 · W50/IBI |
| 정규화 진폭비 | 3 | RI · notch_rel_height · IPA |
| 1차 미분 | 2 | max_slope_norm · t_max_slope_rel |
| 2차 미분 (APG) | 5 | b/a · c/a · d/a · e/a · aging_index |
| 검출 플래그 | 2 | notch_found · apg_found |

문헌 기준점 16개 중 **스케일 불변**인 것만 남겼다. 절대 진폭, 경직도지수(신장 필요),
PTT(ECG 필요), 3차 미분(125 Hz에서 신뢰 불가)은 제외했다.

## 환자 단위 43개

- 중앙값 **20** — 박동별 특징의 환자 내 중앙값
- 박동간 IQR **20** — 원단위. `IQR/|중앙값|`은 부호가 바뀌는 특징에서 발산한다
- 검출률 **3** — `cd_rate` · `ri_rate` · `lvet_rate`. 계산 실패를 버리지 않고 특징으로 올린다

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import numpy as np
import pandas as pd

from ppg_fm import paths, features as F

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)
print("프로젝트", paths.ROOT)
print("데이터  ", paths.data_root())
from ppg_fm.report import Report
rep = Report("01_dataset/04_waveform_features")
print("산출물 →", rep.dir)

In [ ]:
for name, fs in F.FAMILY.items():
    print(f"{name:14s} {len(fs):2d}  {' · '.join(fs)}")
print()
print(f"박동 단위 {len(F.BEAT)}개 → 환자 단위 {len(F.PATIENT)}개")

## 1. 박동 특징 추출

계획표의 세그먼트를 읽어 박동 단위 특징을 뽑는다. 증분 저장이라 중단해도 이어서 돌릴 수 있다.
전 코호트는 수십 분 걸린다.

In [ ]:
from ppg_fm.data import extract

beats_path = paths.interim("beat_features_v2.csv")
if not beats_path.exists():
    while True:
        r = extract.run(budget_sec=600)
        print(r)
        if r["status"] == "done" or r.get("remaining", 0) == 0:
            break
print("추출 완료:", beats_path)

In [ ]:
B = pd.read_csv(beats_path, usecols=["subject"] + F.MORPH,
                dtype={"subject": str}, nrows=200_000)
print(f"박동 {len(B):,} (일부) · 환자 {B.subject.nunique():,}")
B[F.MORPH].describe().T.round(3)

## 2. 검출률 — 결측을 정보로

중복절흔 소실과 c·d파 융합은 잡음에서도, 동맥 경직에서도 나타난다.
결측으로 버리면 병든 환자를 체계적으로 배제하게 되므로 환자별 검출 비율을 특징으로 올린다.

In [ ]:
det = B[F.MORPH].notna().mean().sort_values()
rep.table(det.rename("beat_detection_rate").rename_axis("feature").reset_index(), "beat_detection_rate.csv", "박동 단위 검출률")
det.head(8).round(3).to_frame("박동 검출률")

## 3. 환자 요약 — 43개

In [ ]:
from ppg_fm.data import summary

feat_path = paths.interim("patient_features_v2.csv")
if not feat_path.exists():
    summary.build()
P = pd.read_csv(feat_path, dtype={"subject": str})
print(f"환자 {len(P):,} · 요약 특징 {len(F.PATIENT)}개")
desc = P[F.PATIENT + ["HR", "n_beats"]].describe().T.reset_index().rename(columns={"index": "feature"})
rep.table(desc, "patient_feature_summary.csv", "환자 요약 특징 43개 분포")
P[F.RATE + ["n_beats", "HR"]].describe().round(3)

## 4. 재현성 — 측정이 안정적인가

같은 환자의 박동을 홀수·짝수로 나눠 두 번 계산했을 때 두 값이 일치하는지를 본다.
중앙값 계열 0.97–0.99, 검출률 0.99 이상, IQR 계열 0.95 이상이면 세그먼트를 더 늘려 얻을 것이 없다.

In [ ]:
rel = rep.path("reliability_projection.csv")
if rel.exists():
    display(pd.read_csv(rel).head(12))
else:
    print("재현성 산출은 02_logistic/01_summary_method.ipynb 에서 다룬다")

## 산출물

In [ ]:
rep.done("박동 특징 23개 추출과 환자 요약 43개")
rep.summary()